# Line Segmentation and Visualization
Given a page image, segment it into lines.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path("../src").resolve()))

In [2]:
import json
import subprocess
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import fitz
import os
import torch
from dataset import AlignCollate
from model import Model
from llama_cpp import Llama
from huggingface_hub import hf_hub_download
from utils import CTCLabelConverter

/home/tomas/projects/renAIssance/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

def render_pdf_page(pdf_path, page_num, out_dir="tmp", dpi=400):
    """
    Renders a single page from a PDF to a PNG image.
    Args:
        pdf_path (str or Path): Path to the PDF file.
        page_num (int): 1-based page number to render.
        out_dir (str or Path): Directory to save the image.
        dpi (int): Render DPI.
    Returns:
        Path to the saved PNG image.
    """
    pdf_path = Path(pdf_path).expanduser().resolve()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    scale = dpi / 72.0
    mat = fitz.Matrix(scale, scale)
    with fitz.open(pdf_path) as doc:
        if not (1 <= page_num <= doc.page_count):
            raise ValueError(f"Page number {page_num} out of range (1-{doc.page_count})")
        page = doc.load_page(page_num - 1)
        pix = page.get_pixmap(matrix=mat, alpha=False)
        out_path = out_dir / f"{pdf_path.stem}_page{page_num}.png"
        pix.save(str(out_path))
    return out_path

In [4]:
# Example usage:
pdf_file = "/mnt/c/Users/tomas/renAIssance/original_data/Print/Buendia - Instruccion.pdf"  # Set your PDF path here
page_number = 3  # Set the page number you want to render
img_path = render_pdf_page(pdf_file, page_number)
print(f"Saved page {page_number} as image: {img_path}")

Saved page 3 as image: tmp/Buendia - Instruccion_page3.png


In [5]:
def run_kraken_segment(png_path, out_json, text_direction='horizontal-lr'):
    cmd = [
        'kraken',
        '-i', str(png_path),
        str(out_json),
        'segment',
        '-bl',
        '-d', text_direction,
    ]
    subprocess.run(cmd, check=True)

def bbox_from_boundary(boundary):
    pts = np.asarray(boundary, dtype=np.float32)
    xs = pts[:, 0]
    ys = pts[:, 1]
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())

def extract_lines_from_kraken_json(seg):
    lines = seg.get('lines', [])
    out = []
    for i, ln in enumerate(lines):
        bbox = None
        if isinstance(ln, dict):
            if 'bbox' in ln and ln['bbox'] is not None:
                bb = ln['bbox']
                if isinstance(bb, (list, tuple)) and len(bb) == 4:
                    bbox = tuple(map(int, bb))
            elif 'boundary' in ln and ln['boundary'] is not None:
                bbox = bbox_from_boundary(ln['boundary'])
        if bbox is None:
            continue
        out.append({'i': i, 'bbox': bbox, 'raw': ln})
    return out

In [6]:
# --- Segment page and save crops ---
page_path = Path('/home/tomas/projects/renAIssance/notebooks/tmp/Buendia - Instruccion_page3.png')
output_dir = Path('output_lines')
output_dir.mkdir(exist_ok=True)
seg_json = output_dir / 'segmentation.json'

run_kraken_segment(page_path, seg_json)
seg = json.loads(seg_json.read_text(encoding='utf-8'))
lines = extract_lines_from_kraken_json(seg)

page_img = Image.open(page_path).convert('L')
crops = []
for i, ln in enumerate(lines, start=1):
    x1, y1, x2, y2 = ln['bbox']
    crop = page_img.crop((x1, y1, x2, y2))
    crops.append((crop, f'line_{i:04d}'))  # label is dummy

# --- Build a simple dataset and dataloader using AlignCollate ---
class SimpleLineDataset(torch.utils.data.Dataset):
    def __init__(self, crops):
        self.crops = crops
    def __len__(self):
        return len(self.crops)
    def __getitem__(self, idx):
        img, label = self.crops[idx]
        return img, label

opt = type('opt', (), {})()
opt.imgH = 64  # Set to your model's training height

collate_fn = AlignCollate(imgH=opt.imgH)
dataset = SimpleLineDataset(crops)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

Loading ANN /home/tomas/projects/renAIssance/.venv/lib/python3.10/site-packages/kraken/blla.mlmodel	✓
Segmenting /home/tomas/projects/renAIssance/notebooks/tmp/Buendia - Instruccion_page3.png	✓


# Load pretrained model

Load a pretrained model for inference

In [7]:
import torch
from types import SimpleNamespace
from model import Model

# Load checkpoint
ckpt_path = "/home/tomas/projects/renAIssance/renAIssance/ytlxnj3i/checkpoints/epoch=90-step=5642.ckpt"
ckpt = torch.load(ckpt_path, map_location="cpu")

# Extract hyperparameters and build opt
hparams = ckpt["hyper_parameters"]
opt = SimpleNamespace(**hparams)
converter = CTCLabelConverter(opt.character)
opt.num_class = len(converter.character)  # Ensure num_class is set correctly

# Remove 'model.' prefix from state_dict keys if present
state_dict = ckpt["state_dict"]
new_state_dict = {k.replace('model.', ''): v for k, v in state_dict.items()}

# Instantiate model
model = Model(opt)
model.load_state_dict(new_state_dict)

model.eval()

No Transformation module specified


Model(
  (FeatureExtraction): ResNet_FeatureExtractor(
    (ConvNet): ResNet(
      (conv0_1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn0_1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv0_2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn0_2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=Tru

In [8]:
output_txt_path = "output_lines/predictions.txt"
predictions = []

with torch.no_grad():
    for images, labels in dataloader:
        if next(model.parameters()).is_cuda:
            images = images.cuda()
        batch_size = images.size(0)
        text_input = torch.zeros((batch_size, opt.batch_max_length + 1), dtype=torch.long)
        if images.device != text_input.device:
            text_input = text_input.to(images.device)
        preds = model(images, text_input)
        _, preds_index = preds.max(2)
        preds_size = torch.IntTensor([preds.size(1)] * batch_size)
        # Use converter.decode as in training/validation
        preds_str = converter.decode(preds_index.data, preds_size.data)
        for label, pred in zip(labels, preds_str):
            print(f"{label}: {pred}")
            predictions.append(f"{label}: {pred}")

with open(output_txt_path, "w", encoding="utf-8") as f:
    for line in predictions:
        f.write(line + "\n")

print(f"Predictions saved to {output_txt_path}")

line_0001: ruc. ibid.
line_0002: Psal. Ri. g.-
line_0003: ooTí8. Iso
line_0004: c jó. 8.
line_0005: matth Iy.
line_0006: i4.-
line_0007: Azrci, ro.
line_0008: rr-
line_0009: Matr. I8.-
line_0010: r. Cc.-
line_0011: guro disseño de su edad: la Reli-
line_0012: gion para con Dios en la devota
line_0013: assistencia a los Templos; la piedad
line_0014: con los Dadres en la obediéncia-
line_0015: mas rendida; y la modestia, y de-
line_0016: seo de saber, con los mayores,
line_0017: gustando mas de oir, y pregun-
line_0018: tar, que de definir, y resolver. Bién
line_0019: que esto en vuestra infinita Sabi-
line_0020: duria fue soberana dignacion, y
line_0021: en la natural ignorancia de los
line_0022: Niños es indispensable necessi-
line_0023: dad.-
line_0024: Ni tienen solamente en Vos
line_0025: el disseño, la luz, y el exemplo,
line_0026: sino tambien el amor, y protec-
line_0027: cion. Vos, como singular Maes-
line_0028: tro de los Niños, les dais enten-
line_0029: dimiento, y comunicais

# LLM-based post-processing

In [9]:
model_path = hf_hub_download(
    repo_id="Qwen/Qwen2.5-3B-Instruct-GGUF",
    filename="qwen2.5-3b-instruct-q4_k_m.gguf",
)

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=8,
)

llama_model_loader: loaded meta data with 26 key-value pairs and 435 tensors from /home/tomas/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct-GGUF/snapshots/7dabda4d13d513e3e842b20f0d435c732f172cbe/qwen2.5-3b-instruct-q4_k_m.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = qwen2.5-3b-instruct
llama_model_loader: - kv   3:                            general.version str              = v0.1-v0.1
llama_model_loader: - kv   4:                           general.finetune str              = qwen2.5-3b-instruct
llama_model_loader: - kv   5:                         general.size_label str              = 3.4B
llama_model_loade

In [17]:
SYSTEM_PROMPT = """Estás asistiendo en la corrección posterior a OCR de textos históricos en español.

Recibirás una página completa de texto obtenida mediante OCR.
Cada línea del input tiene este formato exacto:

line_0001: texto OCR
line_0002: texto OCR
...

Tu tarea es reconstruir la lectura más probable del texto original usando el contexto de toda la página y devolverlo en español moderno.

Objetivo principal:
- interpretar correctamente el contenido,
- corregir errores evidentes del OCR,
- resolver palabras deformadas o mal reconocidas,
- y producir una lectura fluida, coherente y plausible en español moderno.

Convenciones a tener en cuenta:
- Las letras "u" y "v" pueden aparecer intercambiadas. Interprétalas según la ortografía moderna.
- Las letras "f" y "s" pueden aparecer intercambiadas. Interprétalas según la ortografía moderna.
- Los acentos son inconsistentes. Ignóralos excepto la "ñ", que debe conservarse.
- Algunas letras presentan una pequeña barra horizontal. Esto suele indicar que sigue una "n", o que tras una "q" sigue "ue".
- La "ç" debe interpretarse siempre como "z" en español moderno.

Restricciones de formato obligatorias:
- Devuelve EXACTAMENTE el mismo número de líneas que en la entrada.
- Conserva EXACTAMENTE cada identificador de línea (por ejemplo, "line_0001:").
- No añadas comentarios, explicaciones ni texto adicional.

La salida debe seguir exactamente este formato:
line_0001: texto corregido
line_0002: texto corregido
"""

In [18]:
# Join OCR predictions into one page text
page_text = "\n".join(predictions)

# Call the LLM
response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Texto OCR:\n\n{page_text}"}
    ],
    temperature=0.2
)

corrected_text = response["choices"][0]["message"]["content"].strip()


# Show result in notebook
print("\n\n----- LLM OUTPUT -----\n")
print(corrected_text)


# Save result
output_llm_path = "output_lines/predictions_llm.txt"

with open(output_llm_path, "w", encoding="utf-8") as f:
    f.write(corrected_text)

print(f"\nLLM output saved to {output_llm_path}")

Llama.generate: 151 prefix-match hit, remaining 1447 prompt tokens to eval
llama_perf_context_print:        load time =   37542.80 ms
llama_perf_context_print: prompt eval time =   31199.48 ms /  1447 tokens (   21.56 ms per token,    46.38 tokens per second)
llama_perf_context_print:        eval time =   95016.20 ms /  1023 runs   (   92.88 ms per token,    10.77 tokens per second)
llama_perf_context_print:       total time =  128790.12 ms /  2470 tokens
llama_perf_context_print:    graphs reused =        990




----- LLM OUTPUT -----

line_0001: Ruc. Ibíd.
line_0002: Psal. Ri. g.
line_0003: Oto. Isó.
line_0004: C jó. 8.
line_0005: Matth Iy.
line_0006: 14.
line_0007: Azrci, ro.
line_0008: r.
line_0009: Matr. I8.
line_0010: r. Cc.
line_0011: Guro diseno de su edad: la Religión para con Dios en la devota
line_0012: asistencia a los Templos; la piedad
line_0013: con los Padres en la obediencia-
line_0014: mas rendida; y la modestia, y de-
line_0015:seo de saber, con los mayores,
line_0016: gustando mas de oir, y preguntar, que de definir, y resolver. Bien
line_0017: que esto en vuestra infinita Sabiduría fue soberana dignación, y
line_0018: en la natural ignorancia de los
line_0019: Niños es indispensable necesidad.
line_0020: Ni tienen solamente en Vos
line_0021: el diseno, la luz, y el ejemplo,
line_0022: sino tambien el amor, y protección.
line_0023: Vos, como singular Maestro de los Niños, les dais entendimiento, y
line_0024: comunicais la sabiduría. Vos les prometeis el Reyno
line_0025: de